# 02 — Data pipeline sanity check

Visual companion to `scripts/sanity_check.py` (the authoritative, assertion-based
check — run that first; it must exit 0 before any model code is written).

Here we *look* at the pipeline: a denormalized augmented batch, the eval batch,
and per-split class distributions. The heavy augmentation is our main mitigation
for PlantVillage's background-bias problem, so it's worth eyeballing.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset import PlantVillageDataset
from src.data.loaders import build_dataset, build_loader
from src.data.splits import LabelMaps, load_split
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD
from src.utils.seed import seed_everything

seed_everything(42)

# Edit to your local data location.
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "plantvillage dataset" / "color"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
maps = LabelMaps.from_json(SPLITS_DIR / "label_maps.json")
print("species:", maps.num_species, "| classes:", maps.num_classes)

## Denormalize helper

In [ ]:
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

def denorm(t: torch.Tensor) -> np.ndarray:
    img = (t * std + mean).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

## Heavy train augmentation — many draws of one image

Same source image, 8 augmented draws. They should vary substantially (crop,
flip, color, occlusion) — that variation is what discourages background memorization.

In [ ]:
train_ds = build_dataset(data_root=DATA_ROOT, splits_dir=SPLITS_DIR, split="train",
                         mode="flat", img_size=224, aug_strength="heavy")
idx = 0
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.ravel():
    img, label = train_ds[idx]
    ax.imshow(denorm(img)); ax.axis("off")
    ax.set_title(maps.id_to_class[label].replace("___", "\n"), fontsize=7)
fig.suptitle("Heavy train augmentation — 8 draws of the same image")
plt.tight_layout(); plt.show()

## Eval batch (minimal transform)

In [ ]:
val_ds = build_dataset(data_root=DATA_ROOT, splits_dir=SPLITS_DIR, split="val",
                       mode="flat", img_size=224)
loader = build_loader(val_ds, batch_size=8, is_train=False, num_workers=0)
xb, yb = next(iter(loader))
print("batch:", tuple(xb.shape), "labels:", yb.tolist())

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, label in zip(axes.ravel(), xb, yb):
    ax.imshow(denorm(img)); ax.axis("off")
    ax.set_title(maps.id_to_class[int(label)].replace("___", "\n"), fontsize=7)
plt.tight_layout(); plt.show()

## Per-split class distribution

Confirms the stratified split kept proportions stable across train/val/test.

In [ ]:
import pandas as pd
frames = {s: load_split(SPLITS_DIR, s) for s in ("train", "val", "test")}
dist = pd.DataFrame({s: f["class_name"].value_counts() for s, f in frames.items()}).fillna(0)
props = dist.div(dist.sum(axis=1), axis=0)
ax = props.sort_index().plot(kind="barh", stacked=True, figsize=(8, 11))
ax.set(title="Per-class split proportions (approx 0.70/0.15/0.15 everywhere)", xlabel="proportion")
plt.tight_layout(); plt.show()
props.describe().loc[["mean", "min", "max"]].round(3)

## Weighted sampler check

Draw many indices through the `WeightedRandomSampler` and confirm it flattens the
(otherwise ~36x) class imbalance toward uniform.

In [ ]:
from src.training.samplers import make_weighted_sampler
labels = frames["train"]["class_id"].to_numpy()
sampler = make_weighted_sampler(labels, maps.num_classes)
drawn = np.array([labels[i] for i in list(sampler)[:30000]])
orig = np.bincount(labels, minlength=maps.num_classes)
samp = np.bincount(drawn, minlength=maps.num_classes)
print(f"original imbalance: {orig.max()/orig.min():.1f}x")
print(f"sampled  imbalance: {samp.max()/max(samp.min(),1):.1f}x  (should be near 1x)")